# Phase 3 — Causal Forest on a downsample

**Goal.** Add the Causal Forest to the model roster. Unlike the Phase 2
learners, it can't easily eat 11M training rows on a MacBook Air — the
random-forest structure holds pointers into the training set per tree, so
memory scales with (n_estimators × n_train). We train it on a **stratified
500k downsample** of the training data and evaluate on the **same 2.8M
held-out test set** as Phase 2. That keeps the Qini comparison honest:
every model is scored on identical test data; only the training set
differs.

**What Causal Forest is.** A random forest whose splits maximize
*treatment-effect heterogeneity* rather than outcome variance. Each leaf
ends up with users whose treatment responses are similar; the predicted
uplift for a new user is the average treatment effect in the leaves it
lands in. Signature property: an asymptotic-normal theory that gives
per-user confidence intervals on the predicted uplift — rare and valuable.

**Fair-comparison caveat.** The Causal Forest sees ~1/22 the training
data of the Phase 2 models. If it wins anyway, that's a strong result.
If it loses, we can't fully separate "the algorithm is weaker on this
data" from "it just had less data." We'll flag this in the writeup.

In [ ]:
import sys
import time
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.uplift.data import load_criteo, stratified_downsample, FEATURE_COLS
from src.uplift.split import make_split
from src.uplift.evaluation import qini_curve, qini_coefficient
from src.uplift.plots import plot_qini_curves
from src.uplift.learners import (
    PropensityBaseline,
    SLearner,
    TLearner,
    ClassTransformationLearner,
    XLearner,
    CausalForestLearner,
)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")

## 1. Load, split, downsample training

Same 80/20 split as Phase 2 (seed 42, stratified on treatment) so the
test set is byte-identical. Then we take a **stratified 500k sample from
train** — preserving the 85/15 treatment ratio — as the Causal Forest's
training data. The other five models see the full 11.2M for a
"best available performance per algorithm" comparison.

In [ ]:
df = load_criteo()
train, test = make_split(df)

# 500k stratified sample of train, preserving the 85/15 treatment ratio
train_small = stratified_downsample(train, n=500_000, random_state=42)
print(f"train (full):  {len(train):>10,}   treated share: {train['treatment'].mean():.1%}")
print(f"train (500k):  {len(train_small):>10,}   treated share: {train_small['treatment'].mean():.1%}")
print(f"test:          {len(test):>10,}   treated share: {test['treatment'].mean():.1%}")
print(f"test conversions: {test['conversion'].sum():,}")

## 2. Prepare arrays

In [ ]:
# Full-train arrays for Phase 2 models
X_tr = train[FEATURE_COLS].to_numpy()
T_tr = train["treatment"].to_numpy()
Y_tr = train["conversion"].to_numpy()

# 500k downsample arrays for Causal Forest
X_tr_small = train_small[FEATURE_COLS].to_numpy()
T_tr_small = train_small["treatment"].to_numpy()
Y_tr_small = train_small["conversion"].to_numpy()

# Held-out test arrays (same for every model)
X_te = test[FEATURE_COLS].to_numpy()
T_te = test["treatment"].to_numpy()
Y_te = test["conversion"].to_numpy()

print(f"train (full): {X_tr.shape}, train (500k): {X_tr_small.shape}, test: {X_te.shape}")

## 3. Fit all six models

Same five Phase 2 models on the full training set. Then Causal Forest on
the 500k downsample. Expect the CF to be the slowest by a wide margin —
even on 500k it's a proper forest with 200 trees per estimator.

| model | training rows | rough time |
|---|---|---|
| Propensity, S, T, ClassTrans | 11.2M | ~1 min each |
| X-learner | 11.2M (5 sub-fits) | ~3 min |
| Causal Forest | 500k | **5–15 min** on M2 |

Total budget: ~15–25 min. If the CF takes too long, cut `n_estimators`
from 200 to 100 in the constructor.

In [ ]:
models = {
    "Propensity (baseline)": (PropensityBaseline(), "full"),
    "S-learner":              (SLearner(),           "full"),
    "T-learner":              (TLearner(),           "full"),
    "Class Transformation":   (ClassTransformationLearner(), "full"),
    "X-learner":              (XLearner(),           "full"),
    "Causal Forest":          (CausalForestLearner(n_estimators=200), "500k"),
}

fit_times = {}
for name, (m, which) in models.items():
    Xt, Tt, Yt = (X_tr, T_tr, Y_tr) if which == "full" else (X_tr_small, T_tr_small, Y_tr_small)
    t0 = time.time()
    m.fit(Xt, Tt, Yt)
    fit_times[name] = time.time() - t0
    print(f"  {name:<25s} ({which:>4s} train, {len(Xt):>9,} rows) fit in {fit_times[name]:6.1f}s")

## 4. Predict on the shared test set + Qini curves

In [ ]:
curves = {}
coefs = {}
for name, (m, _) in models.items():
    u = m.predict_uplift(X_te)
    curve = qini_curve(Y_te, T_te, u)
    curves[name] = curve
    coefs[name] = qini_coefficient(curve)

coef_df = pd.DataFrame({
    "model": list(coefs.keys()),
    "train_size": [models[n][1] for n in coefs.keys()],
    "qini_coefficient": list(coefs.values()),
    "fit_time_s": [fit_times[n] for n in coefs.keys()],
}).sort_values("qini_coefficient", ascending=False).reset_index(drop=True)

coef_df.style.format({"qini_coefficient": "{:+.2f}", "fit_time_s": "{:.1f}"}) \
    .bar(subset=["qini_coefficient"], color="#b6532c", align=0)

## 5. Qini curves — 6 models on the same test set

In [ ]:
ordered = dict(sorted(curves.items(), key=lambda kv: -coefs[kv[0]]))

fig, ax = plt.subplots(figsize=(9, 5.5))
plot_qini_curves(ordered, ax=ax, title="Qini curves — 6 models, Criteo held-out 2.8M test set")
plt.tight_layout()
plt.show()

## 6. Top-K targeting — the business number

In [ ]:
def top_k_lift(curve: pd.DataFrame, k_frac: float) -> float:
    idx = (curve["share"] >= k_frac).idxmax()
    return float(curve["qini"].iloc[idx])

total_qini = curves["S-learner"]["qini"].iloc[-1]

rows = []
for name, curve in curves.items():
    row = {"model": name}
    for frac in [0.10, 0.20, 0.50]:
        lift = top_k_lift(curve, frac)
        row[f"lift@{int(frac*100)}%"] = lift
        row[f"share@{int(frac*100)}%"] = lift / total_qini
    rows.append(row)

topk_df = pd.DataFrame(rows).set_index("model").reindex(coef_df["model"])

topk_df.style.format({
    "lift@10%": "{:+.1f}", "lift@20%": "{:+.1f}", "lift@50%": "{:+.1f}",
    "share@10%": "{:.1%}", "share@20%": "{:.1%}", "share@50%": "{:.1%}",
})

## 7. Interpretation

**How to read the Qini plot:**
- Curves *above* the dashed random-targeting line = model finds real persuadables at the top of its ranking.
- Curves *below* the line = model is worse than random (it's putting sleeping dogs or lost causes at the top).
- The steeper the curve rises at the start, the more concentrated the incremental conversions are in the top-ranked users — which is exactly what a marketer wants.

**Things to look for in your results:**
1. Does the Causal Forest beat / tie / lose to the Phase 2 models despite the 22× smaller training set? Any of these outcomes is defensible in the writeup.
2. Does the X-learner still lead among the full-data models? If so, the story is clean: the imbalance-handling estimator wins on this imbalanced dataset.
3. Where does the propensity baseline land? If it's competitive on top-10%, that's genuinely surprising and worth calling out honestly.

**Phase 4 next:**
- Bootstrap the Qini coefficient (200 resamples) so the ranking has 95% CIs — the current point estimates could be within noise of each other.
- Calibration check: bin predicted uplift and compare against observed uplift per bin.
- AUUC as a second summary metric.
- README writeup with the winning-model headline and Qini figure.